## Data Cleaning

In [1]:
import pandas as pd

In [2]:
# Load the raw transactions data
transactions = pd.read_csv("../data/raw/train_transaction.csv")

In [5]:
# Create a copy of the dataset and remove duplicate rows
cleaned = transactions.copy()
cleaned.drop_duplicates(inplace=True)

In [6]:
# Check that the duplicates have been removed
duplicate_count = cleaned.duplicated().sum()
duplicate_count

np.int64(0)

In [7]:
# Separate the target variable from the features
target = cleaned["isFraud"]
features = cleaned.drop(columns=["isFraud"])

In [8]:
# Remove the transaction ID column from features as it is irrelevant for the model (but keep it for later use)
transaction_ids = features["TransactionID"]
features = features.drop(columns=["TransactionID"])

In [10]:
# Identify columns with majority of missing values (more that 80%)
missing_percent = features.isnull().mean() * 100

high_missing_columns = missing_percent[
    missing_percent > 80
].sort_values(ascending=False)

high_missing_columns

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D9       87.312290
D8       87.312290
V138     86.123717
V139     86.123717
V140     86.123717
V141     86.123717
V142     86.123717
V147     86.123717
V146     86.123717
V148     86.123717
V149     86.123717
V158     86.123717
V157     86.123717
V156     86.123717
V155     86.123717
V154     86.123717
V153     86.123717
V163     86.123717
V161     86.123717
V162     86.123717
V151     86.122701
V143     86.122701
V144     86.122701
V150     86.122701
V145     86.122701
V152     86.122701
V159     86.122701
V160     86.122701
V164     86.122701
V165     86.122701
V166     86.122701
V322     86.054967
V323     86.054967
V324     86.054967
V325     86.054967
V326     86.054967
V327     86.054967
V328     86.054967
V329     86.054967
V330     86.054967
V331     86.054967
V332     86.054967
V333     86.054967
V334     86.054967
V335     86.054967
V336     86.054967
V337     86.

In [11]:
# Drop columns with more than 80% missing values
columns_to_drop = high_missing_columns.index.tolist()

features = features.drop(columns=columns_to_drop)

In [13]:
# Check the shape of the cleaned features dataset
features.shape

(590540, 337)

In [15]:
# Separate the columns into numerical and categorical
numerical_columns = features.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_columns = features.select_dtypes(include=["object", "category", "string"]).columns.tolist()

In [17]:
# Print the number of numerical and categorical columns
print("Numerical columns:", len(numerical_columns))
print("Categorical Columns: ", len(categorical_columns))

Numerical columns: 323
Categorical Columns:  14


In [18]:
# Fill in the missing values in the numerical columns (median fill)
for column in numerical_columns:
    features[column] = features[column].fillna(
        features[column].median()
    )

In [19]:
# Fill in the missing values in the categorical columns
for column in categorical_columns:
    features[column] = features[column].fillna("Unknown")

In [20]:
# Confirm that there are no more missing values
features.isnull().sum().sum()

np.int64(0)

In [21]:
# Rebuild the cleaned dataset by combining the features and target variable
cleaned_data = features.copy()

# Reinsert the TransactionID column at the beginning of the cleaned dataset
cleaned_data.insert(
    loc=0, column="TransactionID", value=transaction_ids
)

cleaned_data["isFraud"] = target.values

In [22]:
cleaned_data.head()

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,V313,V314,V315,V316,V317,V318,V319,V320,V321,isFraud
0,2987000,86400,68.5,W,13926,361.0,150.0,discover,142.0,credit,...,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0
1,2987001,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2987002,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2987003,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,...,0.0,0.0,0.0,50.0,1404.0,790.0,0.0,0.0,0.0,0
4,2987004,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [23]:
cleaned_data.shape

(590540, 339)

In [26]:
# Save the cleaned dataset to a new .csv file
cleaned_data.to_csv(
    "../data/processed/train_transaction_cleaned.csv", index=False
)

KeyboardInterrupt: 

In [27]:
cleaned_data.to_parquet(
    "../data/processed/train_transaction_cleaned.parquet",
    index=False
)


ArrowKeyError: A type extension with name pandas.period already defined